# Topic Modeling of Recent Academic Publications using FASTopic
## Discovering Hot Research Topics from 2020 to 2025

In [ ]:
%%capture
!pip install fastopic
!pip install sentence-transformers
!pip install plotly
!pip install scikit-learn

In [ ]:
!pip install -qq numpy==1.26.4 gensim
get_ipython().kernel.do_shutdown(restart=True)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. Dataset

In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/MachineLearning/processed.csv")
print(df.shape)
df.head()

In [ ]:
df = df.dropna(subset=['combined_lemmatized'])

target_sample_size = 20000
year_counts = df['year'].value_counts().sort_index()
print("Original year distribution:")
print(year_counts)

total_docs = len(df)
sample_ratio = target_sample_size / total_docs

sampled_dfs = []
for year in sorted(df['year'].unique()):
    year_df = df[df['year'] == year]
    year_sample_size = int(len(year_df) * sample_ratio)
    year_sample_size = max(year_sample_size, 100)

    if len(year_df) >= year_sample_size:
        year_sample = year_df.sample(n=year_sample_size, random_state=42)
    else:
        year_sample = year_df

    sampled_dfs.append(year_sample)
    print(f"Year {year}: {len(year_df)} → {len(year_sample)} documents")

df_sample = pd.concat(sampled_dfs, ignore_index=True)
df_sample = df_sample.sort_values('year').reset_index(drop=True)

documents = df_sample['combined_lemmatized'].tolist()
years = df_sample["year"].astype(str).tolist()

print(f"\nFinal sample: {len(documents)} documents")
print("Sample year distribution:")
print(df_sample['year'].value_counts().sort_index())
print("First document sample:")
print(documents[0][:300], "...")

## 2. Modeling

In [ ]:
from fastopic import FASTopic
from topmost.preprocess import Preprocess

stopwords = ['model', 'models', 'learn', 'learning', 'network', 'networks',
    'performance', 'training', 'train', 'trained', 'test', 'testing',
    'method', 'methods', 'approach', 'approaches', 'algorithm', 'algorithms',
    'system', 'systems', 'framework', 'data', 'dataset', 'evaluation',
    'experiment', 'experiments', 'analysis', 'result', 'results']

preprocess = Preprocess(vocab_size=10000, stopwords=stopwords)

topic_model = FASTopic(
    num_topics=50,
    preprocess=preprocess,
    doc_embed_model="paraphrase-mpnet-base-v2",
    num_top_words=15,
    low_memory=True,
    low_memory_batch_size=2000,
    normalize_embeddings=True,
    verbose=False
)

#### 2.1 Training

In [ ]:
import time

start = time.time()
top_words, doc_topic_dist = topic_model.fit_transform(documents)
end = time.time()
duration = end - start
print(f"Training completed in {duration:.2f} seconds ({duration/60:.2f} minutes).")

In [ ]:
save_path = "/content/drive/MyDrive/fastopic_model.zip"
topic_model.save(save_path)
print("Model saved to Google Drive.")

In [ ]:
print("Number of topics discovered:", len(topic_model.get_top_words()))
print("\nTop words for each topic:")
for i in range(len(topic_model.get_top_words())):
    topic_words = topic_model.get_topic(topic_idx=i)
    words = [word for word, _ in topic_words[:10]]
    print(f"Topic {i}: {', '.join(words)}")

## 3. Visualizations

In [ ]:
topic_model.visualize_topic(top_n=10)

In [ ]:
topic_model.visualize_topic_hierarchy()

In [ ]:
fig = topic_model.visualize_topic_weights(top_n=20, height=500)
fig.show()

### Topics over Time - Growing Topics from 2020 to 2025

In [ ]:
time_slices = [int(year) for year in years]
topic_activity = topic_model.topic_activity_over_time(time_slices)
fig = topic_model.visualize_topic_activity(
    top_n=10,
    topic_activity=topic_activity,
    time_slices=time_slices
)
fig.show()

In [ ]:
def analyze_top_growing_topics(topic_activity, time_slices, top_n=10):
    unique_years = sorted(set([int(year) for year in time_slices]))
    n_topics, n_years = topic_activity.shape

    print(f"Topics: {n_topics}, Years: {n_years}")
    print(f"Unique years: {unique_years}")

    yearly_activity = pd.DataFrame(
        topic_activity.T,
        index=unique_years,
        columns=[f"Topic_{i}" for i in range(n_topics)]
    )

    first_year = yearly_activity.index.min()
    last_year = yearly_activity.index.max()
    growth = yearly_activity.loc[last_year] - yearly_activity.loc[first_year]

    top_growing = growth.sort_values(ascending=False).head(top_n)

    return top_growing, yearly_activity

top_growing, yearly_activity = analyze_top_growing_topics(topic_activity, time_slices)

print("Top Growing Topics from 2020 to 2025:")
for i, (topic_col, growth_val) in enumerate(top_growing.items(), 1):
    topic_idx = int(topic_col.split('_')[1])
    topic_words = topic_model.get_topic(topic_idx=topic_idx)
    words = [word for word, _ in topic_words[:5]]
    print(f"{i}. Topic {topic_idx}: {', '.join(words)} (Growth: {growth_val:.4f})")

#### Growing Topics from 2020 to 2025

| Topic ID | Description                                                              |
|----------|--------------------------------------------------------------------------|
| 22       | Large Language Models and Their Reasoning Abilities          |
| 2        | Visual and Video Data Using Multimodal and Semantic Methods  |
| 44       | Image Generation Using Diffusion-Based Techniques             |
| 7        | Object Detection and Scene Segmentation in Images                        |
| 32       | Development and Design Challenges in Software Engineering Tools          |
| 39       | Reinforcement Learning and Agent-Environment Interaction                 |
| 37       | Discussions Around Trust, Legal, and Ethical Issues in AI                |
| 45       | Problems Related to Model Degradation and Incremental Updates            |
| 18       | Exploration of New Research Areas and Evolving Scientific Landscapes     |
| 26       | Model Compression, Token Processing, and Distillation     |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics.pairwise import cosine_similarity

beta_matrix = topic_model.get_beta()
print(f"Beta matrix shape: {beta_matrix.shape}")

topic_similarity = cosine_similarity(beta_matrix)
print(f"Topic similarity matrix shape: {topic_similarity.shape}")

In [ ]:
sns.heatmap(topic_similarity,
            annot=False,
            center=0.5,
            square=True,
            linewidths=0.05,
            vmin=0,
            vmax=0.4,
            cbar_kws={"shrink": .8, "label": "Cosine Similarity"})

plt.title('Similarity Matrix',
          fontsize=14, fontweight='bold')
plt.xlabel('Topic Index', fontsize=12)
plt.ylabel('Topic Index', fontsize=12)
plt.tight_layout()
plt.savefig('/content/drive/MyDrive/fastopic_similarity_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

mask = ~np.eye(len(topic_similarity), dtype=bool)
print(f"Average topic similarity: {topic_similarity[mask].mean()}")
print(f"Max topic similarity: {topic_similarity[mask].max()}")
print(f"Min topic similarity: {topic_similarity[mask].min()}")

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
from sklearn.manifold import TSNE

tsne = TSNE(n_components=2, random_state=42, perplexity=30)
doc_2d = tsne.fit_transform(doc_topic_dist)
dominant_topics = np.argmax(doc_topic_dist, axis=1)
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=doc_2d[:, 0],
    y=doc_2d[:, 1],
    mode='markers',
    marker=dict(
        color=dominant_topics,
        size=5,
        opacity=0.7,
        colorscale='Viridis',
        showscale=True,
        colorbar=dict(title="Topic ID")
    ),
    text=[f"Doc {i}<br>Topic: {dominant_topics[i]}" for i in range(len(doc_2d))],
    hovertemplate='<b>%{text}</b><br>(%{x:.2f}, %{y:.2f})<extra></extra>',
    name='Documents'
))

fig.update_layout(
   title={
       'text': '<b>Documents and Topics</b>',
       'x': 0.5,
       'font': {'size': 20}
   },
   xaxis_title='Dimension 1',
   yaxis_title='Dimension 2',
   width=800,
   height=600
)

fig.show()

print("Topic Keywords:")
for i in range(10):
    topic_words = topic_model.get_topic(topic_idx=i)
    words = [word for word, _ in topic_words[:5]]
    print(f"Topic {i}: {', '.join(words)}")

## 4. Performance Metrics

### 4.1. Coherence Scores

In [ ]:
from gensim.models.coherencemodel import CoherenceModel
from gensim.corpora import Dictionary

tokenized_docs = [doc.split() for doc in documents]
dictionary = Dictionary(tokenized_docs)
corpus = [dictionary.doc2bow(text) for text in tokenized_docs]

topics_for_coherence = []
top_words_data = topic_model.get_top_words()
for topic_words in top_words_data:
    words = topic_words if isinstance(topic_words, list) else topic_words.split()
    topics_for_coherence.append(words)

coherence_cv = CoherenceModel(topics=topics_for_coherence, texts=tokenized_docs, dictionary=dictionary, coherence='c_v').get_coherence()
coherence_umass = CoherenceModel(topics=topics_for_coherence, texts=tokenized_docs, dictionary=dictionary, coherence='u_mass').get_coherence()
coherence_npmi = CoherenceModel(topics=topics_for_coherence, texts=tokenized_docs, dictionary=dictionary, coherence='c_npmi').get_coherence()
coherence_uci = CoherenceModel(topics=topics_for_coherence, texts=tokenized_docs, dictionary=dictionary, coherence='c_uci').get_coherence()

print(f"C_v Coherence:     {coherence_cv:.4f}")
print(f"U_Mass Coherence: {coherence_umass:.4f}")
print(f"NPMI Coherence:   {coherence_npmi:.4f}")
print(f"UCI Coherence:    {coherence_uci:.4f}")

### 4.2. PUW

In [ ]:
all_words = [word for topic in topics_for_coherence for word in topic]
unique_words = set(all_words)
puw = len(unique_words) / len(all_words)

print(f"Proportion of Unique Words (PUW): {puw:.4f}")

### 4.3. Avg. Jaccard Similarity

In [ ]:
from itertools import combinations

def jaccard_similarity(set1, set2):
    return len(set1 & set2) / len(set1 | set2)

jaccard_scores = []
for t1, t2 in combinations(topics_for_coherence, 2):
    jaccard_scores.append(jaccard_similarity(set(t1), set(t2)))

avg_jaccard = sum(jaccard_scores) / len(jaccard_scores)
print(f"Average Jaccard Similarity between topics: {avg_jaccard:.4f}")